# Insmile AI — Fine-Tune Qwen2-VL-7B for Dental X-Ray Analysis

This notebook fine-tunes **Qwen2-VL-7B** on the **DENTEX** dataset using **QLoRA** (4-bit quantization + LoRA adapters) to fit within Google Colab's free T4 GPU (16GB VRAM).

**What this produces:** A LoRA adapter (~150MB) that makes the model significantly better at:
- Detecting dental caries, deep caries, periapical lesions, and impacted teeth
- Providing accurate FDI tooth numbers
- Generating precise bounding boxes
- Outputting structured JSON

## Instructions
1. Open this in Google Colab (GPU runtime → T4)
2. Run all cells in order
3. Training takes ~3-5 hours on a free T4
4. The adapter auto-uploads to HuggingFace when done

---

## 1. Setup — Install Dependencies

In [ ]:
%%capture
# IMPORTANT: After this cell runs, restart runtime (Runtime → Restart runtime)
# then re-run all cells. The Pillow downgrade requires a fresh Python process.

!pip install "pillow<11" --force-reinstall --quiet
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers accelerate bitsandbytes peft trl datasets
!pip install -U qwen-vl-utils
!pip install -U huggingface_hub scipy

# Auto-restart the runtime so the Pillow fix takes effect
import os
os.kill(os.getpid(), 9)

## 2. Configuration

In [ ]:
import os
from google.colab import userdata

# ============================================================
# CONFIGURATION — Edit these values
# ============================================================

# HuggingFace token — set in Colab secrets (Key icon on left sidebar → add HF_TOKEN)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError(
        'HF_TOKEN not found! Add it in Colab: Left sidebar → Key icon → '
        'New secret → Name: HF_TOKEN, Value: your hf_... token'
    )

# Model
BASE_MODEL = 'Qwen/Qwen2-VL-7B-Instruct'  # Instruct version for better JSON output

# Where to save the fine-tuned adapter on HuggingFace
HF_REPO_NAME = 'joshuarebo/insmile-dental-vision-lora'

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 1              # Limited by VRAM
GRAD_ACCUM_STEPS = 8       # Effective batch size = 1 * 8 = 8
LEARNING_RATE = 2e-4       # Standard for QLoRA
MAX_SEQ_LENGTH = 2048      # Max tokens per example
LORA_R = 16                # LoRA rank (higher = more capacity, more VRAM)
LORA_ALPHA = 32            # LoRA scaling factor
LORA_DROPOUT = 0.05

print(f'Model: {BASE_MODEL}')
print(f'Output repo: {HF_REPO_NAME}')
print(f'Epochs: {EPOCHS}, LR: {LEARNING_RATE}, LoRA rank: {LORA_R}')

## 3. Login to HuggingFace

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

## 4. Download DENTEX Dataset from HuggingFace

**Source:** DENTEX Challenge 2023 (MICCAI) — `ibrahimhamamci/DENTEX`  
**Content:** 1,005 fully annotated panoramic X-rays with bounding boxes  
**Classes:** Caries, Deep Caries, Periapical Lesions, Impacted Teeth  
**Format:** COCO JSON annotations  
**License:** CC BY-NC-SA 4.0

We use `snapshot_download()` to get the raw files directly (NOT `load_dataset()`).

In [ ]:
import json
import os
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET_DIR = Path('/content/DENTEX')

# Download the DENTEX dataset using snapshot_download (downloads raw files)
print('Downloading DENTEX dataset from HuggingFace...')
print('This downloads ~3.9GB of panoramic X-ray images + COCO annotations.')
print('='*60)

snapshot_download(
    repo_id="ibrahimhamamci/DENTEX",
    repo_type="dataset",
    local_dir=str(DATASET_DIR),
)

print(f'\n✅ Download complete: {DATASET_DIR}')

# Show what we got
print('\nDirectory structure:')
for item in sorted(DATASET_DIR.iterdir()):
    if item.name.startswith('.'):
        continue
    if item.is_dir():
        sub_files = list(item.rglob('*'))
        sub_files = [f for f in sub_files if f.is_file()]
        images = [f for f in sub_files if f.suffix.lower() in ('.png', '.jpg', '.jpeg')]
        jsons = [f for f in sub_files if f.suffix.lower() == '.json']
        print(f'  📁 {item.name}/ — {len(images)} images, {len(jsons)} JSONs, {len(sub_files)} total files')
    else:
        print(f'  📄 {item.name} ({item.stat().st_size/1024:.1f}KB)')

# Find all annotation JSON files
all_jsons = [f for f in DATASET_DIR.rglob('*.json') if not f.name.startswith('.')]
print(f'\nAll JSON annotation files:')
for jf in sorted(all_jsons):
    size = jf.stat().st_size / 1024
    print(f'  {jf.relative_to(DATASET_DIR)} ({size:.1f}KB)')
    # Peek at structure
    try:
        with open(jf) as f:
            data = json.load(f)
        if isinstance(data, dict):
            keys = list(data.keys())
            print(f'    Keys: {keys}')
            if 'images' in data:
                print(f'    Images: {len(data["images"])}')
            if 'annotations' in data:
                print(f'    Annotations: {len(data["annotations"])}')
            if 'categories' in data:
                cats = {c['id']: c['name'] for c in data['categories']}
                print(f'    Categories: {cats}')
    except Exception as e:
        print(f'    (Could not parse: {e})')

# Find all images
all_images = list(DATASET_DIR.rglob('*.png')) + list(DATASET_DIR.rglob('*.jpg'))
print(f'\nTotal images: {len(all_images)}')
if all_images:
    img_dirs = sorted(set(p.parent for p in all_images))
    for d in img_dirs[:10]:
        count = len([f for f in all_images if f.parent == d])
        print(f'  {d.relative_to(DATASET_DIR)}: {count} images')

## 5. Parse DENTEX Annotations → Training Format

We convert the COCO-format annotations into instruction-tuning examples:
- **Input**: dental X-ray image + analysis prompt
- **Output**: structured JSON with findings, tooth numbers, severity, bounding boxes

In [ ]:
import json
import glob
from pathlib import Path
from PIL import Image

# DENTEX category mapping
DENTEX_CATEGORIES = {
    1: {'label': 'Caries', 'severity': 'moderate'},
    2: {'label': 'Deep caries', 'severity': 'severe'},
    3: {'label': 'Periapical lesion', 'severity': 'severe'},
    4: {'label': 'Impacted tooth', 'severity': 'moderate'},
}

# FDI quadrant mapping for tooth number estimation from bbox position
def estimate_fdi_from_bbox(bbox_norm, img_width, img_height):
    """Estimate FDI tooth number from bbox position on a panoramic X-ray.
    Panoramic X-rays are mirrored: patient's right is on image left.
    """
    cx = bbox_norm[0] + bbox_norm[2] / 2  # center x (0-1)
    cy = bbox_norm[1] + bbox_norm[3] / 2  # center y (0-1)
    
    # Determine quadrant
    is_upper = cy < 0.5
    is_right_side = cx < 0.5  # Image left = patient's right
    
    if is_upper and is_right_side:
        quadrant = 1
    elif is_upper and not is_right_side:
        quadrant = 2
    elif not is_upper and not is_right_side:
        quadrant = 3
    else:
        quadrant = 4
    
    # Estimate tooth position (1-8) based on distance from midline
    dist_from_center = abs(cx - 0.5) * 2  # 0 = midline, 1 = edge
    tooth_num = min(8, max(1, int(dist_from_center * 8) + 1))
    
    return f'{quadrant}{tooth_num}'


def parse_dentex_annotations(json_path, images_dir):
    """Parse COCO-format DENTEX annotations into training examples."""
    with open(json_path) as f:
        data = json.load(f)
    
    # Build image lookup
    images = {img['id']: img for img in data.get('images', [])}
    
    # Build category lookup
    categories = {}
    for cat in data.get('categories', []):
        cat_id = cat['id']
        cat_name = cat['name'].lower()
        if 'deep' in cat_name or 'caries' in cat_name and 'deep' in cat_name:
            categories[cat_id] = {'label': 'Deep caries', 'severity': 'severe'}
        elif 'caries' in cat_name:
            categories[cat_id] = {'label': 'Caries', 'severity': 'moderate'}
        elif 'periapical' in cat_name:
            categories[cat_id] = {'label': 'Periapical lesion', 'severity': 'severe'}
        elif 'impacted' in cat_name:
            categories[cat_id] = {'label': 'Impacted tooth', 'severity': 'moderate'}
        else:
            categories[cat_id] = {'label': cat['name'], 'severity': 'moderate'}
    
    # If no categories in file, use defaults
    if not categories:
        categories = DENTEX_CATEGORIES
    
    # Group annotations by image
    img_annotations = {}
    for ann in data.get('annotations', []):
        img_id = ann['image_id']
        if img_id not in img_annotations:
            img_annotations[img_id] = []
        img_annotations[img_id].append(ann)
    
    examples = []
    for img_id, anns in img_annotations.items():
        if img_id not in images:
            continue
        
        img_info = images[img_id]
        img_path = images_dir / img_info['file_name']
        if not img_path.exists():
            # Try common alternatives
            for alt in [images_dir / Path(img_info['file_name']).name,
                        images_dir.parent / 'xrays' / img_info['file_name']]:
                if alt.exists():
                    img_path = alt
                    break
            else:
                continue
        
        img_w = img_info.get('width', 1900)
        img_h = img_info.get('height', 950)
        
        findings = []
        for ann in anns:
            cat_id = ann.get('category_id', 1)
            cat = categories.get(cat_id, {'label': 'Abnormality', 'severity': 'moderate'})
            
            # COCO bbox format: [x, y, width, height] in pixels
            bbox = ann.get('bbox', [0, 0, 100, 100])
            bbox_norm = [
                round(bbox[0] / img_w, 4),
                round(bbox[1] / img_h, 4),
                round(bbox[2] / img_w, 4),
                round(bbox[3] / img_h, 4),
            ]
            
            # Estimate tooth number from position
            tooth = estimate_fdi_from_bbox(bbox_norm, img_w, img_h)
            
            findings.append({
                'label': f"{cat['label']} on tooth {tooth}",
                'tooth': tooth,
                'severity': cat['severity'],
                'confidence': 0.92,
                'bbox_norm': bbox_norm,
            })
        
        if not findings:
            continue
        
        # Build the target JSON
        target = {
            'findings': findings[:6],
            'overall': f"Panoramic X-ray showing {len(findings)} pathological finding{'s' if len(findings) > 1 else ''}: {', '.join(set(c['label'].split(' on ')[0] for c in findings[:4]))}.",
            'confidence': 0.88,
            'recommendations': generate_recommendations(findings),
            'image_quality': 'good',
        }
        
        examples.append({
            'image_path': str(img_path),
            'target_json': json.dumps(target, indent=None),
        })
    
    return examples


def generate_recommendations(findings):
    """Generate clinical recommendations based on findings."""
    recs = []
    labels = [f['label'].split(' on ')[0].lower() for f in findings]
    
    if 'deep caries' in labels:
        recs.append('Urgent: Root canal treatment or extraction may be needed for deep caries')
    if 'caries' in labels:
        recs.append('Composite or amalgam restoration recommended for carious lesions')
    if 'periapical lesion' in labels:
        recs.append('Periapical pathology detected — consider endodontic evaluation and vitality testing')
    if 'impacted tooth' in labels:
        recs.append('Surgical evaluation recommended for impacted tooth — assess proximity to IAN canal')
    if not recs:
        recs.append('Regular follow-up and preventive care recommended')
    
    recs.append('Patient education on oral hygiene and dietary habits')
    return recs[:4]


print('Annotation parser ready.')

In [ ]:
# Parse DENTEX COCO annotations into training examples
all_examples = []

# Find all COCO-format JSON files with annotations
json_files = [f for f in DATASET_DIR.rglob('*.json') if not f.name.startswith('.')]

print(f'Scanning {len(json_files)} JSON files for COCO annotations...')

for jf in sorted(json_files):
    try:
        with open(jf) as f:
            data = json.load(f)
    except:
        continue

    # Must have both 'images' and 'annotations' keys to be COCO format
    if not isinstance(data, dict):
        continue
    if 'annotations' not in data or 'images' not in data:
        continue
    if len(data['annotations']) == 0:
        continue

    print(f'\n📋 {jf.relative_to(DATASET_DIR)}')
    print(f'   {len(data["images"])} images, {len(data["annotations"])} annotations')

    # Build category lookup
    categories = {}
    for cat in data.get('categories', []):
        categories[cat['id']] = cat['name']
    print(f'   Categories: {categories}')

    # Skip files that only have quadrant/enumeration annotations (no diagnosis)
    cat_names_lower = [v.lower() for v in categories.values()]
    has_diagnosis = any(term in ' '.join(cat_names_lower)
                        for term in ['caries', 'periapical', 'impacted', 'lesion',
                                     'decay', 'cavity', 'abscess'])
    if not has_diagnosis:
        print(f'   ⏭️  Skipping (no diagnosis categories, likely enumeration-only)')
        continue

    # Build image lookup
    images_info = {img['id']: img for img in data['images']}

    # Group annotations by image
    img_anns = {}
    for ann in data['annotations']:
        img_id = ann['image_id']
        if img_id not in img_anns:
            img_anns[img_id] = []
        img_anns[img_id].append(ann)

    # Find image directory — search relative to the JSON file
    found_count = 0
    missed_count = 0

    for img_id, anns in img_anns.items():
        if img_id not in images_info:
            continue
        img_info = images_info[img_id]
        fname = img_info['file_name']

        # Search for the image in multiple locations
        img_path = None
        search_paths = [
            jf.parent / fname,
            jf.parent / Path(fname).name,
            jf.parent / 'images' / fname,
            jf.parent / 'images' / Path(fname).name,
            jf.parent / 'train' / fname,
            jf.parent / 'xrays' / fname,
            jf.parent.parent / fname,
            jf.parent.parent / 'images' / fname,
            jf.parent.parent / 'xrays' / fname,
            jf.parent.parent / 'train' / fname,
        ]

        for candidate in search_paths:
            if candidate.exists():
                img_path = candidate
                break

        # Last resort: search entire dataset dir
        if img_path is None:
            matches = list(DATASET_DIR.rglob(Path(fname).name))
            if matches:
                img_path = matches[0]

        if img_path is None:
            missed_count += 1
            if missed_count <= 3:
                print(f'   ⚠️  Image not found: {fname}')
            continue

        found_count += 1
        img_w = img_info.get('width', 2900)
        img_h = img_info.get('height', 1250)

        findings = []
        for ann in anns:
            bbox = ann.get('bbox', [0, 0, 100, 100])
            cat_id = ann.get('category_id', 0)
            cat_name = categories.get(cat_id, 'Abnormality')

            # Map category to our standard labels
            cat_lower = cat_name.lower()
            if 'deep' in cat_lower and 'caries' in cat_lower:
                label = 'Deep caries'
                severity = 'severe'
            elif 'deep caries' in cat_lower:
                label = 'Deep caries'
                severity = 'severe'
            elif 'caries' in cat_lower or 'cavity' in cat_lower or 'decay' in cat_lower:
                label = 'Caries'
                severity = 'moderate'
            elif 'periapical' in cat_lower or 'lesion' in cat_lower:
                label = 'Periapical lesion'
                severity = 'severe'
            elif 'impacted' in cat_lower:
                label = 'Impacted tooth'
                severity = 'moderate'
            else:
                label = cat_name
                severity = 'moderate'

            # Normalize bbox to 0-1
            bbox_norm = [
                round(bbox[0] / img_w, 4),
                round(bbox[1] / img_h, 4),
                round(bbox[2] / img_w, 4),
                round(bbox[3] / img_h, 4),
            ]

            tooth = estimate_fdi_from_bbox(bbox_norm, img_w, img_h)
            findings.append({
                'label': f'{label} on tooth {tooth}',
                'tooth': tooth,
                'severity': severity,
                'confidence': 0.92,
                'bbox_norm': bbox_norm,
            })

        if not findings:
            continue

        target = {
            'findings': findings[:6],
            'overall': f"Panoramic X-ray showing {len(findings)} pathological finding{'s' if len(findings) > 1 else ''}: {', '.join(set(f['label'].split(' on ')[0] for f in findings[:4]))}.",
            'confidence': 0.88,
            'recommendations': generate_recommendations(findings),
            'image_quality': 'good',
        }

        all_examples.append({
            'image_path': str(img_path),
            'target_json': json.dumps(target, indent=None),
        })

    print(f'   ✅ Matched {found_count} images, missed {missed_count}')

# Final report
print(f'\n{"="*60}')
if all_examples:
    print(f'✅ Total training examples: {len(all_examples)}')
    print(f'Sample image: {all_examples[0]["image_path"]}')
    print(f'Sample target:\n{all_examples[0]["target_json"][:400]}')
else:
    print('❌ No training examples found!')
    print('Check the output above — likely the images are in a different')
    print('subdirectory than expected. Look at the directory structure from')
    print('cell 8 output and report back.')
    # Don't raise — let the user see the output and decide

## 6. Build Training Dataset

Convert to the chat format that Qwen2-VL expects for instruction tuning.

In [ ]:
import random
from datasets import Dataset

# System prompt (same as our production prompt, so the model learns our exact format)
SYSTEM_PROMPT = """You are an expert dental radiologist AI. Analyze the dental X-ray and return a JSON object with your findings.

Output format:
{"findings": [{"label": "specific finding", "tooth": "FDI number", "severity": "mild|moderate|severe", "confidence": 0.0-1.0, "bbox_norm": [x, y, w, h]}], "overall": "summary", "confidence": 0.0-1.0, "recommendations": ["action"], "image_quality": "good|fair|poor"}

Rules: bbox_norm values are 0.0-1.0 (normalized). Use FDI tooth numbering. Return JSON ONLY."""

USER_PROMPT = "Analyze this dental radiograph. Identify all visible pathology using FDI tooth numbering. Return ONLY the JSON object."

def build_conversation(example):
    """Build a chat conversation for training."""
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': example['image_path']},
                    {'type': 'text', 'text': USER_PROMPT},
                ]
            },
            {'role': 'assistant', 'content': example['target_json']},
        ]
    }

# Shuffle and split
random.seed(42)
random.shuffle(all_examples)

split_idx = max(1, int(len(all_examples) * 0.9))
train_examples = all_examples[:split_idx]
val_examples = all_examples[split_idx:] if split_idx < len(all_examples) else all_examples[-1:]

train_conversations = [build_conversation(ex) for ex in train_examples]
val_conversations = [build_conversation(ex) for ex in val_examples]

print(f'Training examples: {len(train_conversations)}')
print(f'Validation examples: {len(val_conversations)}')

# Create HuggingFace datasets
train_dataset = Dataset.from_list(train_conversations)
val_dataset = Dataset.from_list(val_conversations)

print(f'\nDataset ready.')
if len(train_dataset) > 0:
    print(f'Sample user prompt: {train_dataset[0]["messages"][1]["content"][1]}')
    print(f'Sample target (first 200 chars): {train_dataset[0]["messages"][2]["content"][:200]}')

## 7. Load Model with QLoRA (4-bit Quantization)

This loads the 7B model in 4-bit precision (~4GB VRAM) and attaches trainable LoRA adapters.

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading model in 4-bit...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print(f'Model loaded. Parameters: {model.num_parameters():,}')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# LoRA configuration — attach adapters to attention layers
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # Attention
        'gate_proj', 'up_proj', 'down_proj',       # MLP
    ],
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print(f'GPU memory after LoRA: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 8. Training with SFTTrainer

Using TRL's SFTTrainer which handles the vision-language chat format automatically.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='/content/insmile-dental-checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field='',
    dataset_kwargs={'skip_prepare_dataset': True},
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to='none',
)

print('Training configuration ready.')
print(f'  Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'  Total training steps: ~{len(train_conversations) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM_STEPS)}')

In [ ]:
from functools import partial
from qwen_vl_utils import process_vision_info

def collate_fn(examples, processor):
    """Custom collator for vision-language training."""
    texts = []
    image_inputs = []
    
    for example in examples:
        messages = example['messages']
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        
        images, videos = process_vision_info(messages)
        image_inputs.append(images)
    
    batch = processor(
        text=texts,
        images=image_inputs[0] if image_inputs[0] else None,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors='pt',
    )
    
    batch['labels'] = batch['input_ids'].clone()
    return batch

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=partial(collate_fn, processor=processor),
    processing_class=processor,
    max_seq_length=MAX_SEQ_LENGTH,
)

print('Trainer initialized. Ready to train.')

In [ ]:
# ============================================================
# TRAIN! This takes 3-5 hours on a T4 GPU.
# ============================================================
print('Starting training...')
print('='*60)

train_result = trainer.train()

print('='*60)
print('Training complete!')
print(f'  Loss: {train_result.training_loss:.4f}')
print(f'  Runtime: {train_result.metrics["train_runtime"]/3600:.1f} hours')
print(f'  Samples/sec: {train_result.metrics["train_samples_per_second"]:.2f}')

## 9. Save & Upload Adapter to HuggingFace

In [ ]:
# Save the LoRA adapter locally
ADAPTER_PATH = '/content/insmile-dental-adapter'
model.save_pretrained(ADAPTER_PATH)
processor.save_pretrained(ADAPTER_PATH)

print(f'Adapter saved to {ADAPTER_PATH}')

# Check adapter size
import os
total_size = sum(os.path.getsize(os.path.join(ADAPTER_PATH, f))
                 for f in os.listdir(ADAPTER_PATH)
                 if os.path.isfile(os.path.join(ADAPTER_PATH, f)))
print(f'Adapter size: {total_size / 1e6:.1f} MB')

In [ ]:
# Upload to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()

# Create repo if it doesn't exist
try:
    api.create_repo(HF_REPO_NAME, private=True, exist_ok=True)
except Exception as e:
    print(f'Repo creation note: {e}')

# Upload adapter
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=HF_REPO_NAME,
    commit_message='Insmile dental vision LoRA adapter - trained on DENTEX',
)

print(f'\nAdapter uploaded to: https://huggingface.co/{HF_REPO_NAME}')
print('\nYou can now use this adapter in your Insmile backend!')

## 10. Quick Validation — Test the Fine-Tuned Model

In [ ]:
# Test on a validation image
if val_examples:
    test_example = val_examples[0]
    test_image = Image.open(test_example['image_path']).convert('RGB')
    
    test_messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': test_image},
                {'type': 'text', 'text': USER_PROMPT},
            ]
        },
    ]
    
    text = processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
    images, _ = process_vision_info(test_messages)
    
    inputs = processor(
        text=[text], images=images, return_tensors='pt', padding=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1000, temperature=0.1)
    
    response = processor.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    
    print('MODEL OUTPUT:')
    print(response[:500])
    print('\n---\nEXPECTED:')
    print(test_example['target_json'][:500])
else:
    print('No validation examples available for testing.')

## Done!

Your fine-tuned LoRA adapter is now on HuggingFace. Next steps:

1. **Deploy**: Host the model on RunPod Serverless or HuggingFace Inference Endpoints
2. **Integrate**: Update `server/src/services/openrouter.js` to call your self-hosted model
3. **Iterate**: As dentists use the app and correct findings, save those corrections as new training data

---
*Generated by Insmile AI training pipeline*